# Strategic Swarm Operator Surface

Operator notebook for the **Strategic Swarm Agent** — a structural fragility reasoning engine.

Run this notebook top-to-bottom to:
1. Inspect provider health and check which data sources are configured
2. Execute a live, latest-window, demo, or replay run
3. Inspect synthesis signals injected by the web search enricher
4. Read the full rendered Markdown report

---

## Modes

| Mode | Description | Required parameters |
|---|---|---|
| `live` | Fetch live signals between an explicit start/end timestamp | `LIVE_START`, `LIVE_END` |
| `latest` | Fetch live signals for the last N minutes from now | `LOOKBACK_MINUTES` |
| `demo` | Run a canned scenario from sample data — works without any API keys | `SCENARIO` |
| `replay` | Re-run reasoning on previously stored raw signals | `REPLAY_RUN_ID` **or** `LIVE_START`/`LIVE_END` |

## Web search enrichment

When `OPENAI_API_KEY` is configured, the pipeline automatically enriches the top clusters
(those with ≥ 3 signals by default, up to 5 per run) with live web synthesis via
OpenAI's `web_search_preview` tool **after** clustering and pattern matching.

One targeted fragility question is derived per qualifying cluster. The resulting synthesis
signals appear in the provenance trail and in the **Synthesis signals** section below.
The same `OPENAI_API_KEY` also enables LLM-backed refinement in pattern matching, fragility
scoring, and report generation.

---

**Quick start for live mode** — set keys in `.env` then run all cells:
```
NEWSAPI_API_KEY=...
ALPHAVANTAGE_API_KEY=...
FRED_API_KEY=...
OPENAI_API_KEY=...      # optional: enables LLM refinement + web search enrichment
```

## 1 · Setup

Import the runner and helper functions. The runner auto-loads `.env` and `.env.local` from
the repository root via `bootstrap_env()`, so you only need to set keys once in your `.env`
file — no need to set them here.

In [2]:
from __future__ import annotations

import os
from datetime import UTC, datetime, timedelta
from pathlib import Path
from pprint import pprint

from strategic_swarm_agent.graph.runner import StrategicSwarmRunner
from strategic_swarm_agent.presentation.operator_surface import (
    available_demo_scenarios,
    list_recent_runs,
    run_and_summarize,
)

OUTPUT_DIR = Path("outputs")
runner = StrategicSwarmRunner(output_dir=OUTPUT_DIR)
print("Runner ready.")

Runner ready.


## 2 · Provider health

Check which live data providers are configured. A provider marked `configured: False` will
be skipped silently during live/latest runs — only the providers with keys present will
fetch data.

| Provider | Source family | What it fetches |
|---|---|---|
| `newsapi` | `news` | News articles via NewsAPI `/everything` + `/top-headlines` |
| `alphavantage` | `market` | Market news sentiment and global equity/ETF quotes |
| `fred` | `macro` | Federal Reserve macro series (10Y yield, HY spread, SOFR, Fed balance sheet) |
| `gdelt` | `alt` | GDELT DOC alternative structural events |

**GDELT** is the only provider that does not require a key — it always shows `configured: True`.

Web search enrichment status is shown separately since it is not a standard provider.

In [3]:
# Live provider health
health = runner.provider_health()
for row in health:
    status = "✓" if row["configured"] else "✗"
    print(f"  {status} {row['provider_name']:16s}  [{row['source_family']}]")

# Web search enricher (uses OPENAI_API_KEY, not a standard provider)
web_search_ok = bool(os.getenv("OPENAI_API_KEY"))
llm_ok = bool(os.getenv("OPENAI_API_KEY"))
print()
print(f"  {'✓' if web_search_ok else '✗'} openai-websearch   [synthesis]  — web_search_preview enrichment")
print(f"  {'✓' if llm_ok else '✗'} openai-llm         [reasoning]  — pattern/fragility/report LLM refinement")

  ✓ newsapi           [news]
  ✓ alphavantage      [market]
  ✓ fred              [macro]
  ✓ gdelt             [alt]

  ✓ openai-websearch   [synthesis]  — web_search_preview enrichment
  ✓ openai-llm         [reasoning]  — pattern/fragility/report LLM refinement


## 3 · Available demo scenarios

Demo scenarios use bundled sample data and work offline with no API keys.

In [4]:
scenarios = available_demo_scenarios()
print("Available demo scenarios:")
for s in scenarios:
    print(f"  • {s}")

Available demo scenarios:
  • arctic_cable_bypass
  • debt_defense_spiral
  • open_model_breakout


## 4 · Run parameters

Edit only the block corresponding to your chosen mode. The others are ignored.

### Mode
```
MODE = "live"     # Fetch live signals between LIVE_START and LIVE_END
MODE = "demo"     # Run a canned scenario — works without API keys
MODE = "latest"   # Fetch the last LOOKBACK_MINUTES of signals from now
MODE = "replay"   # Re-run reasoning on previously stored raw signals
```

### Live / Latest parameters
- **`LIVE_END`** — end of the ingestion window. Defaults to now.
- **`LIVE_START`** — start of the ingestion window. Defaults to `LIVE_END - LOOKBACK_MINUTES`.
- **`LOOKBACK_MINUTES`** — used by `latest` mode and as the default window width for `live`.
  Increase this to capture more history (e.g. `240` for a 4-hour window).
  Typical values: `60` (1h), `120` (2h), `480` (8h).

### Demo parameters
- **`SCENARIO`** — one of the scenario IDs printed in section 3 above.
  Options: `arctic_cable_bypass`, `debt_defense_spiral`, `open_model_breakout`.

### Replay parameters
- **`REPLAY_RUN_ID`** — the `run_id` from a previous run (shown in section 7 · Recent runs).
  Leave empty to replay by time window using `LIVE_START`/`LIVE_END` instead.

In [5]:
# ── Choose your mode ─────────────────────────────────────────────────────────
MODE = "live"           # live | demo | latest | replay

# ── Live / Latest ─────────────────────────────────────────────────────────────
LOOKBACK_MINUTES = 60                                                # window width in minutes
LIVE_END   = datetime.now(UTC).replace(microsecond=0, second=0)     # end of window (now)
LIVE_START = LIVE_END - timedelta(minutes=LOOKBACK_MINUTES)         # start of window

# ── Demo ──────────────────────────────────────────────────────────────────────
SCENARIO = "arctic_cable_bypass"    # arctic_cable_bypass | debt_defense_spiral | open_model_breakout

# ── Replay ────────────────────────────────────────────────────────────────────
REPLAY_RUN_ID = ""      # paste a run_id here, or leave empty to replay by LIVE_START/LIVE_END

print(f"Mode:             {MODE}")
print(f"Window:           {LIVE_START.isoformat()} → {LIVE_END.isoformat()}")
print(f"Lookback:         {LOOKBACK_MINUTES} min")
print(f"Demo scenario:    {SCENARIO}")
print(f"Replay run_id:    {REPLAY_RUN_ID or '(none — will use time window)'}")

Mode:             live
Window:           2026-03-09T20:37:00+00:00 → 2026-03-09T21:37:00+00:00
Lookback:         60 min
Demo scenario:    arctic_cable_bypass
Replay run_id:    (none — will use time window)


## 5 · Execute run

Runs the full pipeline:
`ingest → normalize → cluster → pattern match → web search enrich → fragility → ripple → opportunities → critique → report`

For `live` and `latest` modes this makes real HTTP calls to configured providers.
For `demo` mode it uses bundled sample data — safe to run without any keys.

In [6]:
payload = run_and_summarize(
    runner,
    mode=MODE,
    scenario=SCENARIO if MODE == "demo" else None,
    start_at=LIVE_START if MODE in {"live", "replay"} else None,
    end_at=LIVE_END if MODE in {"live", "replay"} else None,
    lookback_minutes=LOOKBACK_MINUTES if MODE == "latest" else None,
    run_id=REPLAY_RUN_ID or None,
)

summary = payload["summary"]
print(f"Run ID:           {summary['run_id']}")
print(f"Publish decision: {summary['publish_decision']}")
print(f"Cluster:          {summary.get('cluster_title', '—')}")
print(f"Fragility score:  {summary.get('fragility_score', '—')}")
print(f"Opportunities:    {summary.get('opportunity_count', 0)}")
if summary.get("monitor_only_reason"):
    print(f"Monitor reason:   {summary['monitor_only_reason']}")

Run ID:           f041269e295b
Publish decision: monitor_only
Cluster:          At risk : The key pipelines , terminals and refineries supplying world energy
Fragility score:  0.37628
Opportunities:    0
Monitor reason:   Only one source family confirmed the story.; Fragility score is below the publication threshold.; No opportunity idea survived execution critique.


## 6 · Run summary

Full summary dict — cluster metadata, topology, fragility score, publish decision,
source counts, and the run directory where outputs were persisted.

In [7]:
pprint(summary)

{'agreement_score': 0.5,
 'cluster_id': '3f78c69e186ff778',
 'cluster_strength': 0.905,
 'cluster_title': 'At risk : The key pipelines , terminals and refineries '
                  'supplying world energy',
 'executive_summary': 'At risk : The key pipelines , terminals and refineries '
                      'supplying world energy is not a simple event cluster. '
                      'It is a empire vs swarm topology in which '
                      'rhyljournal.co.uk represents a costly defensive surface '
                      'and newsshopper.co.uk uses adaptive low-cost pressure '
                      'against a rigid defensive system. to exploit this '
                      'breach: The dominant system cannot cheaply defend every '
                      'exposed node.',
 'fragility_score': 0.37628,
 'monitor_only_reason': 'Only one source family confirmed the story.; '
                        'Fragility score is below the publication threshold.; '
                        'No op

## 7 · Synthesis signals (web search enrichment)

When `OPENAI_API_KEY` is configured and the run mode is `live` or `latest`, the pipeline
queries OpenAI `web_search_preview` once per qualifying cluster and injects the results back
as `RawSignal` records with `source = "synthesis"`.

Each synthesis signal corresponds to one cited source URL from the web search response.
The `query_key` field identifies which cluster triggered the query, giving full provenance.

In [8]:
final_state = payload["result"]["final_state"]

synthesis_signals = [
    s for s in (final_state.get("raw_signals") or [])
    if s.source == "synthesis"
]

print(f"Synthesis signals injected: {len(synthesis_signals)}")
if synthesis_signals:
    print()
    for sig in synthesis_signals:
        print(f"  cluster → {sig.query_key}")
        print(f"  title   : {sig.title}")
        print(f"  url     : {sig.source_url}")
        print(f"  conf    : {sig.confidence}")
        print()

Synthesis signals injected: 4

  cluster → united kingdom::key-pipelines-risk-terminals
  title   : Web synthesis: united kingdom::key-pipelines-risk-terminals
  url     : None
  conf    : 0.82

  cluster → united states::iran-puts-risk-war
  title   : US aircraft carrier arrives in the Middle East as tensions with Iran remain high :: WRAL.com
  url     : https://www.wral.com/news/ap/58e6d-us-aircraft-carrier-arrives-in-the-middle-east-as-tensions-with-iran-remain-high/?utm_source=openai
  conf    : 0.82

  cluster → united states::iran-puts-risk-war
  title   : The Latest: Iran’s top diplomat calls indirect US-Iran talks in Oman a ‘good start’ :: WRAL.com
  url     : https://www.wral.com/news/ap/3e192-the-latest-iran-s-top-diplomat-calls-indirect-us-iran-talks-in-oman-a-good-start/?utm_source=openai
  conf    : 0.82

  cluster → united states::iran-puts-risk-war
  title   : Iran closes its airspace to commercial aircraft for hours as tensions with US remain high :: WRAL.com
  url     

## 8 · Detected Scenario

The `detect_scenario` node synthesises all clusters into a named macro narrative with an ordered **causal consequence chain**.
This chain drives both the targeted web search queries and the equity opportunity mapping.


In [ ]:
scenario = final_state.get("detected_scenario")
if scenario:
    print(f"Scenario : {scenario.scenario_name}")
    print(f"Type     : {scenario.scenario_type}")
    print(f"Confidence: {scenario.confidence:.0%}")
    if scenario.key_actors:
        print(f"Key actors: {', '.join(scenario.key_actors)}")
    if scenario.geographic_scope:
        print(f"Geography : {', '.join(scenario.geographic_scope)}")
    print()
    if scenario.consequence_chain:
        print("Causal Consequence Chain:")
        for i, step in enumerate(scenario.consequence_chain, 1):
            print(f"  {i}. {step}")
else:
    print("No scenario detected (demo mode uses static data without LLM).")


## 9 · Equity Opportunities

The `map_equity_opportunities` node uses the detected scenario's consequence chain to identify specific
publicly-traded companies that benefit (`long`), are hurt (`short`), or merit monitoring (`watch`).
Ticker symbols come from the LLM's knowledge — no static watchlist required.
The top 3 symbols are then confirmed via a targeted web search.


In [ ]:
equity_opps = final_state.get("equity_opportunities") or []
if equity_opps:
    header = f"{'Symbol':<12} {'Company':<30} {'Dir':<7} {'Conf':>5}  Rationale"
    print(header)
    print("-" * len(header))
    for opp in equity_opps:
        arrow = {"long": "⬆ LONG", "short": "⬇ SHORT", "watch": "👁 WATCH"}.get(opp.direction, opp.direction)
        print(f"{opp.symbol:<12} {opp.company_name:<30} {arrow:<7} {opp.confidence:>4.0%}  {opp.rationale[:90]}")
        if opp.search_summary:
            print(f"  ↳ Web: {opp.search_summary[:120]}")
else:
    print("No equity opportunities mapped (demo mode or no scenario detected).")


## 10 · Pipeline state keys

Inspect which state keys are populated after the run. Useful for debugging or for
extracting specific intermediate outputs (clusters, patterns, fragility assessments, etc.).

In [9]:
print("Final state keys:", list(final_state.keys()))
print()

# Quick inspection of intermediate outputs
clusters = final_state.get("event_clusters") or []
patterns = final_state.get("abstract_patterns") or []
fragility = final_state.get("fragility_assessments") or []
ripples   = final_state.get("ripple_scenarios") or []
opps      = final_state.get("reviewed_opportunities") or []

print(f"  Clusters identified:    {len(clusters)}")
print(f"  Abstract patterns:      {len(patterns)}")
print(f"  Fragility assessments:  {len(fragility)}")
print(f"  Ripple scenarios:       {len(ripples)}")
print(f"  Reviewed opportunities: {len(opps)}")

Final state keys: ['run_mode', 'window_start', 'window_end', 'raw_signals', 'normalized_events', 'event_clusters', 'selected_cluster', 'abstract_patterns', 'signal_bundles', 'fragility_assessments', 'ripple_scenarios', 'candidate_opportunities', 'reviewed_opportunities', 'final_report', 'diagnostics', 'provenance', 'opportunity_retry_count', 'max_opportunity_retries']

  Clusters identified:    61
  Abstract patterns:      1
  Fragility assessments:  1
  Ripple scenarios:       1
  Reviewed opportunities: 3


In [10]:
clusters

[EventCluster(cluster_id='3f78c69e186ff778', story_key='united kingdom::key-pipelines-risk-terminals', canonical_title='At risk : The key pipelines , terminals and refineries supplying world energy', summary='20260309T204500Z', region='United Kingdom', language='English', entities=['rhyljournal.co.uk', 'newsshopper.co.uk', 'kilburntimes.co.uk', 'eadt.co.uk', 'wiltshiretimes.co.uk', 'thewestonmercury.co.uk'], tags=['english', 'gdelt', 'rhyljournal.co.uk', 'united-kingdom', 'newsshopper.co.uk', 'kilburntimes.co.uk', 'eadt.co.uk', 'wiltshiretimes.co.uk', 'thewestonmercury.co.uk'], source_families=['alt'], signal_ids=['c307e84b346a8e80', 'c307e84b346a8e80', 'c307e84b346a8e80', 'f0a21665e8802a43', 'f0a21665e8802a43', 'f0a21665e8802a43'], first_seen_at=datetime.datetime(2026, 3, 9, 20, 45, tzinfo=datetime.timezone.utc), last_seen_at=datetime.datetime(2026, 3, 9, 21, 0, tzinfo=datetime.timezone.utc), novelty_score=0.9, agreement_score=0.5, cluster_strength=0.905),
 EventCluster(cluster_id='f8

## 11 · Recent runs

Lists the last few persisted runs from the output directory.
Use a `run_id` from here to populate `REPLAY_RUN_ID` in section 4.

In [ ]:
recent = list_recent_runs(OUTPUT_DIR, limit=10)
if not recent:
    print("No persisted runs found.")
else:
    for run in recent:
        ts = datetime.fromtimestamp(run["updated_at"], tz=UTC).strftime("%Y-%m-%d %H:%M UTC")
        print(f"  [{ts}]  {run['run_id']}  {run['scenario']:25s}  {run['publication_status']}")

## 12 · Recent signals in database

Lists the most recently persisted raw signals. Only populated after at least one live or
latest-window run. Demo signals are not persisted by default.

In [ ]:
signals = runner.list_signals(limit=15)
if not signals:
    print("No signals in database. Run a live or latest-window run to populate.")
else:
    for s in signals[:15]:
        print(f"  [{s.get('provider_name', '?'):18s}]  {s.get('signal_type', '?'):20s}  {s.get('title', '')[:70]}")

## 13 · Rendered report

The full Markdown report generated by the pipeline.

The **Provenance** section at the bottom traces every step, including any synthesis signals
injected by the web search enricher:
```
Web search enriched N cluster(s) → M synthesis signal(s).
```

In [ ]:
report_md = payload.get("report_markdown")
if report_md:
    print(report_md)
else:
    print("No report generated for this run (monitor_only with no publishable thesis).")
    print(f"Provenance trail:")
    for line in (final_state.get("provenance") or []):
        print(f"  • {line}")